# Translated lincRNA ORFs in hepatic cell lines

Automated discovery and prioritization of translated ORFs in de novo annotated lincRNAs of HepG2 / Huh7,
replacing the manual IGV pre-selection used for ZFAS1 and FRG1BP (see [README.md](README.md)).

* **Heavy computations** are done by the Nextflow pipeline [lincorfz](lincorfz/), run as SLURM jobs.
* **This notebook** prepares inputs and parameters, prints the bash/`sbatch` commands, and inspects the results.
  It contains only light-weight code (reading tables, plotting). It can itself be executed as a SLURM job,
  see the last section.
* **All outputs** are written to `<WF_RUNS_DIR>/Liver_lincRNA_ORFs_Claude/`.

Commands printed by this notebook must be executed on a sciCORE **login node** (they only submit SLURM jobs
or install software).

In [ ]:
import os
import re
import json
import glob
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display

sns.set_theme(style="white", font_scale=1.0)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

In [ ]:
# 1. environment variables (shared Liver_projects configuration, symlinked into the repository root)
load_dotenv("../Liver_projects.scicore.env")

subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "ext_data_dir": os.getenv("EXT_DATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
}
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_chrom_sizes_file": os.getenv("HUMAN_CHROM_SIZES_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
}

# 2. project-specific locations
PROJECT_DIR = Path.cwd()                      # .../CRISPR_projects/Liver_lincRNA_ORFs
LINCORFZ_DIR = PROJECT_DIR / "lincorfz"       # Nextflow pipeline
SBATCH_DIR = PROJECT_DIR / "sbatch"

OUT_BASE = Path(subdirs["wf_runs_dir"]) / "Liver_lincRNA_ORFs_Claude"   # ALL outputs go here
INPUTS_DIR = OUT_BASE / "inputs"
PARAMS_DIR = OUT_BASE / "params"
RUNS_DIR = OUT_BASE / "runs"
CONDA_ENVS_DIR = OUT_BASE / "conda_envs"
FIG_DIR = OUT_BASE / "figures"
for d in [INPUTS_DIR, PARAMS_DIR, RUNS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# data sources
RNASEQ_RUN_DIR = Path(subdirs["wf_runs_dir"]) / "Asiers_HepG2_Huh7_RNAseq_v1" / "output"
RIBOSEQ_BAM_DIR = Path(subdirs["lab_group_dir"]) / ("RNA_oxidation/wf_runs/Juan_pre_rRNA/results/"
                   "splicing_adapter_polya4_local_sjdb_annot3_novel8_NM1_all_replicates_v1/resolution")
POLYSOME_DIR = Path(subdirs["wf_runs_dir"]) / "polysome_seq_hepg2_inayat_v1" / "polysome_normalised_biwig_data"
CONSERVATION_DIR = Path(subdirs["ext_data_dir"]) / "conservation_and_mutations"
HCC_ANNOT_DIR = Path(subdirs["tables_dir"]) / "HCC_ORFs" / "annotations"
print("outputs:", OUT_BASE)

## 1. Conda environments

sciCORE compute nodes have no internet access, therefore the two environments of the pipeline
(`lincorfz/envs/*.yml`) are built **once, on a login node**; Nextflow then only activates them inside the
SLURM jobs. The `nextflow` conda environment (Nextflow >= 24) is used to run the pipeline driver.

In [ ]:
print(f"bash {LINCORFZ_DIR}/setup_conda_envs.sh {CONDA_ENVS_DIR}")
print("# if not available yet:  conda create -n nextflow -c conda-forge -c bioconda nextflow")

## 2. Prepare inputs

Sample sheets read by the pipeline:

* `rnaseq.tsv`: HepG2 / Huh7 bulk RNA-seq (PMID 40885748): STAR genome alignments and `SJ.out.tab`
* `riboseq.tsv`: HepG2 CHX-only Ribo-seq, genomic alignments with resolved multimappers
* `polysome.tsv`: HepG2 polysome fractions 80S (k = 1 ribosome) ... decasome (k = 10); the free, 40S, 60S and
  heavysome fractions are not used for the mean ribosome load
* `rnaseq_tracks.tsv`: library-size normalized RNA-seq bigwigs (per cell line and strand), only for the IGV session

In [ ]:
# RNA-seq: sample ids and cell lines from the SRA run table
sra = pd.read_csv(Path(subdirs["metadata_dir"]) / "SraRunTable.PMID40885748.csv")
rnaseq = pd.read_csv(Path(subdirs["metadata_dir"]) / "start_samples.rnaseq.pe.tsv", sep="\t")[["sample"]]
rnaseq = rnaseq.merge(sra[["Run", "cell_line"]].rename(columns={"Run": "sample", "cell_line": "condition"}), on="sample")
rnaseq["bam"] = [str(RNASEQ_RUN_DIR / "samples" / s / "map_genome" / f"{s}.Aligned.sortedByCoord.out.bam") for s in rnaseq["sample"]]
rnaseq["sj_tab"] = [str(RNASEQ_RUN_DIR / "samples" / s / "map_genome" / f"{s}.SJ.out.tab") for s in rnaseq["sample"]]
assert all(os.path.exists(p) for p in list(rnaseq["bam"]) + list(rnaseq["sj_tab"]) + [b + ".bai" for b in rnaseq["bam"]])
rnaseq = rnaseq.sort_values(["condition", "sample"])
rnaseq.to_csv(INPUTS_DIR / "rnaseq.tsv", sep="\t", index=False)
display(rnaseq[["sample", "condition"]])

In [ ]:
# Ribo-seq: HepG2, cycloheximide only, 3 replicates
riboseq = pd.DataFrame({"sample": [f"trplct_{i}_HepG2_CHXonly" for i in (7, 8, 9)]})
riboseq["bam"] = [str(RIBOSEQ_BAM_DIR / f"{s}.resolved.bam") for s in riboseq["sample"]]
assert all(os.path.exists(p) and os.path.exists(p + ".bai") for p in riboseq["bam"])
riboseq.to_csv(INPUTS_DIR / "riboseq.tsv", sep="\t", index=False)
display(riboseq)

In [ ]:
# polysome fractions -> number of ribosomes; strands: forward = plus, reverse = minus
n_ribosomes = {"80s": 1, "disome": 2, "trisome": 3, "tetrasome": 4, "pentasome": 5,
               "hexasome": 6, "heptasome": 7, "octasome": 8, "nonasome": 9, "decasome": 10}
rows = []
for p in sorted(POLYSOME_DIR.glob("*.bw")):
    m = re.match(r"HepG2polysome(.+)_rep(\d+)\.spikeinNorm\.(forward|reverse)\.bw", p.name)
    fraction, rep, s = m.groups()
    if fraction in n_ribosomes:
        rows.append(dict(path=str(p), fraction=fraction, n_ribosomes=n_ribosomes[fraction], replicate=int(rep),
                         strand="plus" if s == "forward" else "minus"))
polysome = pd.DataFrame(rows).sort_values(["n_ribosomes", "replicate", "strand"])
polysome.to_csv(INPUTS_DIR / "polysome.tsv", sep="\t", index=False)
display(polysome.groupby(["n_ribosomes", "fraction"])["replicate"].apply(lambda x: sorted(set(x))).to_frame("replicates"))

In [ ]:
# RNA-seq coverage (library-size normalized, per cell line and strand) for the IGV session
ls_dir = RNASEQ_RUN_DIR / "LS_normalized_genomic_coverage" / "per_etiology_group"
rnaseq_tracks = pd.DataFrame([dict(label=c, strand="+" if s == "plus" else "-", path=str(ls_dir / f"{s}.{c}.bw"))
                              for c in ["HepG2", "Huh7"] for s in ["plus", "minus"]])
assert all(os.path.exists(p) for p in rnaseq_tracks["path"])
rnaseq_tracks.to_csv(INPUTS_DIR / "rnaseq_tracks.tsv", sep="\t", index=False)

## 3. Parameter files

Only inputs and run locations are set here; all analysis parameters keep the defaults documented in
`lincorfz/nextflow.config` unless overridden below. `test_chr20` restricts the analysis to chromosome 20,
which carries both previously selected targets (ZFAS1, FRG1BP).

In [ ]:
common_params = {
    "rnaseq_tsv": str(INPUTS_DIR / "rnaseq.tsv"),
    "riboseq_tsv": str(INPUTS_DIR / "riboseq.tsv"),
    "polysome_tsv": str(INPUTS_DIR / "polysome.tsv"),
    "genome": file_paths["human_genome_file"],
    "chrom_sizes": file_paths["human_chrom_sizes_file"],
    "reference_gtf": file_paths["human_annotation_file"],          # GENCODE v42 comprehensive
    "conservation_dir": str(CONSERVATION_DIR),
    "noncode_gtf": str(HCC_ANNOT_DIR / "NONCODEv6_human_hg38_lncRNA.gtf"),
    "hcc_orfs_gtf": str(HCC_ANNOT_DIR / "nORFs_CDSs.gtf"),          # nORFs of the HCC Ribo-seq paper
    "rnaseq_tracks_tsv": str(INPUTS_DIR / "rnaseq_tracks.tsv"),
    "igv_reference_annotation": str(Path(subdirs["human_annotation_dir"]) / "hg38_v42" / "gencode.v42.annotation.sorted.gtf"),
    "conda_envs_dir": str(CONDA_ENVS_DIR),
}
runs = {
    "test_chr20": {"chromosomes": "chr20"},
    "v1": {"chromosomes": "all"},
}
for run_name, extra in runs.items():
    p = {**common_params, **extra, "rundir": str(RUNS_DIR / run_name)}
    with open(PARAMS_DIR / f"{run_name}.json", "w") as fh:
        json.dump(p, fh, indent=4)
    print(PARAMS_DIR / f"{run_name}.json")

## 4. Run the pipeline

The Nextflow driver runs inside a SLURM job (`sbatch/run_lincorfz.sbatch`, QoS `1day`) and submits every
pipeline task as a separate SLURM job. Re-submitting the same command resumes an interrupted run.

In [ ]:
def launch_command(run_name):
    rundir = RUNS_DIR / run_name
    logs = rundir / "slurm_logs"
    logs.mkdir(parents=True, exist_ok=True)   # SLURM silently fails if the log directory does not exist
    return (f"sbatch --output={logs}/driver_%j.out --error={logs}/driver_%j.err "
            f"{SBATCH_DIR}/run_lincorfz.sbatch {PARAMS_DIR / (run_name + '.json')} {rundir} {LINCORFZ_DIR}")

for run_name in runs:
    print(f"# {run_name}\n{launch_command(run_name)}\n")

In [ ]:
# monitoring (login node)
for run_name in runs:
    rundir = RUNS_DIR / run_name
    print(f"# {run_name}")
    print(f"squeue -u $USER")
    print(f"tail -n 30 {rundir}/.nextflow.log")
    print(f"ls -t {rundir}/slurm_logs | head -2\n")

## 5. Results

Set `RUN_NAME` to the run to inspect. Figures are saved to `<OUT_BASE>/figures/<RUN_NAME>/`.

In [ ]:
RUN_NAME = "v1"
RES = RUNS_DIR / RUN_NAME / "results"
FIG = FIG_DIR / RUN_NAME
FIG.mkdir(parents=True, exist_ok=True)
print(RES, RES.exists())

### 5.1 P-site offsets of genomic Ribo-seq reads

Offsets are calibrated per read class (aligned length x 5' soft clip). The metagene shows P-sites of all
calibrated reads around annotated start and stop codons of canonical CDSs (red: frame 0).

In [ ]:
offsets = pd.concat([pd.read_csv(f, sep="\t").assign(sample=Path(f).name.split(".")[0])
                     for f in sorted(glob.glob(str(RES / "riboseq/calibration/*.offsets.tsv")))])
ok = offsets.loc[offsets["pass"]]
summary = ok.groupby("sample").agg(classes=("offset", "size"), cds_reads=("n_cds", "sum"),
                                   frame0=("frame0", "sum"))
summary["frac_frame0"] = summary["frame0"] / summary["cds_reads"]
display(summary)
display(ok.pivot_table(index=["aligned_len"], columns=["sample", "clip5"], values="offset"))

meta = pd.concat([pd.read_csv(f, sep="\t").assign(sample=Path(f).name.split(".")[0])
                  for f in sorted(glob.glob(str(RES / "riboseq/calibration/*.metagene.tsv")))])
samples = sorted(meta["sample"].unique())
fig, axes = plt.subplots(len(samples), 2, figsize=(13, 2.6 * len(samples)), squeeze=False)
for i, s in enumerate(samples):
    for j, anchor in enumerate(["start", "stop"]):
        m = meta.loc[(meta["sample"] == s) & (meta["anchor"] == anchor) & meta["position"].between(-30, 45)]
        colors = ["tab:red" if p % 3 == 0 else ("tab:blue" if p % 3 == 1 else "tab:green") for p in m["position"]]
        axes[i, j].bar(m["position"], m["psites"], color=colors, width=0.9)
        axes[i, j].set(title=f"{s}: P-sites around {anchor} codon", xlabel="position (nt)", ylabel="P-sites")
fig.tight_layout()
fig.savefig(FIG / "psite_metagene.pdf")

stats = pd.concat([pd.read_csv(f, sep="\t").set_index("metric")["value"].rename(Path(f).name.split(".")[0])
                   for f in sorted(glob.glob(str(RES / "riboseq/psite_stats/*.psite_stats.tsv")))], axis=1)
display(stats.loc[[i for i in stats.index if not i.startswith("psites_chr")]])

### 5.2 Mean ribosome load

Isoform-level MRL (coverage-weighted over all exons) of the de novo lincRNAs compared with
Ensembl-canonical protein-coding mRNAs.

In [ ]:
ref_mrl = pd.read_csv(RES / "polysome_scores/polysome.reference_mRNAs.tsv", sep="\t")
tx_mrl = pd.read_csv(RES / "polysome_scores/polysome.transcripts.tsv", sep="\t")
print(f"mRNAs with MRL: {ref_mrl['MRL'].notna().sum()} / {len(ref_mrl)}, median {ref_mrl['MRL'].median():.2f}")
print(f"lincRNA isoforms with MRL: {tx_mrl['MRL'].notna().sum()} / {len(tx_mrl)}, median {tx_mrl['MRL'].median():.2f}")
display(ref_mrl.loc[ref_mrl["gene_name"].isin(["GAPDH", "ACTB", "EEF1A1", "ALB", "APOA1", "TPT1", "RPL3", "ATF4"]),
                    ["gene_name", "MRL", "polysome_mean_coverage", "frac_polysomal_ge2"]])

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4))
bins = np.linspace(1, 10, 37)
ax.hist(ref_mrl["MRL"].dropna(), bins=bins, density=True, histtype="step", lw=2, label="canonical mRNAs")
ax.hist(tx_mrl["MRL"].dropna(), bins=bins, density=True, histtype="step", lw=2, label="de novo lincRNA isoforms")
ax.set(xlabel="mean ribosome load (80S..decasome)", ylabel="density")
ax.legend()
fig.savefig(FIG / "mrl_distributions.pdf", bbox_inches="tight")

### 5.3 De novo lincRNA isoforms

Reasons for (non-)selection of the assembled transcripts that do not encode annotated proteins.

In [ ]:
cand = pd.read_csv(RES / "lincRNAs/lincRNA_candidates.all.tsv", sep="\t", low_memory=False)
sel = pd.read_csv(RES / "lincRNAs/lincRNA_transcripts.tsv", sep="\t", low_memory=False)
display(cand["selection"].value_counts().to_frame("transcripts"))
sel["category"] = np.select([sel["lncRNA_gene_names"].notna(), sel["pseudogene_names"].notna()],
                            ["GENCODE lncRNA", "pseudogene"], "novel")
print(f"{len(sel)} isoforms in {sel['gene_id'].nunique()} loci")
display(pd.crosstab(sel["category"], sel["n_exons"].clip(upper=5).astype(str).replace("5", ">=5")))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for c in ["GENCODE lncRNA", "pseudogene", "novel"]:
    v = np.log10(sel.loc[sel["category"] == c, "TPM_max"])
    axes[0].hist(v, bins=40, histtype="step", lw=2, label=f"{c} (n={len(v)})")
axes[0].set(xlabel="log10 TPM (best cell line)", ylabel="isoforms")
axes[0].legend()
axes[1].hist(sel["tx_length"].clip(upper=10000), bins=50, color="grey")
axes[1].set(xlabel="isoform length (nt, clipped at 10 kb)", ylabel="isoforms")
fig.tight_layout()
fig.savefig(FIG / "lincRNA_isoforms.pdf")

### 5.4 Expression of lincRNA loci (pySanity)

In [ ]:
expr = pd.read_csv(RES / "expression/lincRNA_loci.expression.tsv", sep="\t")
fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))
ax.scatter(expr["log2CPM_HepG2"], expr["log2CPM_Huh7"], s=4, alpha=0.5, color="k")
lim = [np.nanmin(expr[["log2CPM_HepG2", "log2CPM_Huh7"]].values), np.nanmax(expr[["log2CPM_HepG2", "log2CPM_Huh7"]].values)]
ax.plot(lim, lim, ls="--", color="grey")
ax.set(xlabel="HepG2 log2 CPM (pySanity)", ylabel="Huh7 log2 CPM (pySanity)", title=f"{len(expr)} lincRNA loci")
fig.savefig(FIG / "lincRNA_expression_HepG2_vs_Huh7.pdf", bbox_inches="tight")

### 5.5 ORFs: evidence classes and ranking

`ribo_evidence`: strong > moderate > weak > multimapping > none (see README). *multimapping* marks ORFs whose
P-sites come mostly (> 90%) from multimapping reads, typically processed pseudogenes whose Ribo-seq reads stem
from the parent gene. The relationship between read uniqueness and pseudogene annotation is shown below.

In [ ]:
orfs = pd.read_csv(RES / "final/lincORFs.ranked.tsv", sep="\t", low_memory=False)
display(orfs["ribo_evidence"].value_counts().to_frame("ORFs"))
display(orfs.groupby("ribo_evidence")["locus_id"].nunique().to_frame("loci"))
cols = ["rank", "orf_label", "ribo_evidence", "priority_score", "chrom", "orf_start0", "orf_end1", "strand", "n_aa",
        "start_codon", "kozak_context", "MRL", "MRL_pct_vs_mRNA", "log2CPM_HepG2", "log2CPM_Huh7", "pooled_f0",
        "pooled_frac_inframe", "pooled_p_periodicity", "n_reps_p05", "island_log2ratio", "frac_psites_unique",
        "ribo_evidence_all_reads", "ribo_evidence_unique_reads", "PhyloCSF_Median", "lncRNA_gene_names",
        "pseudogene_names", "pseudogene_types", "orf_overlaps_small_RNA", "HCC_paper_same_stop"]
cols = [c for c in cols if c in orfs.columns]
display(orfs[cols].head(40))

In [ ]:
orfs["annotation"] = np.select([orfs["pseudogene_types"].astype(str).str.contains("processed_pseudogene")
                                & ~orfs["pseudogene_types"].astype(str).str.contains("unprocessed"),
                                orfs["pseudogene_names"].notna(), orfs["lncRNA_gene_names"].notna()],
                               ["processed pseudogene", "other pseudogene", "GENCODE lncRNA"], "novel")
ev_all = orfs.loc[orfs["ribo_evidence_all_reads"] != "none"]
fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
sns.histplot(data=ev_all, x="frac_psites_unique", hue="annotation", bins=20, multiple="stack", ax=ax)
ax.set(xlabel="fraction of ORF P-sites from uniquely mapped reads", ylabel="ORFs with Ribo-seq evidence (all reads)")
fig.savefig(FIG / "psite_uniqueness_by_annotation.pdf", bbox_inches="tight")

comp = ["c_mrl", "c_ribo", "c_island", "c_expr", "c_cons", "c_unique"]
ev = orfs.loc[orfs["ribo_evidence"].isin(["strong", "moderate", "weak"])]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(ev[comp].corr(method="spearman"), annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[0])
axes[0].set(title="score components (Spearman, ORFs with Ribo-seq evidence)")
for e in ["strong", "moderate", "weak"]:
    axes[1].hist(orfs.loc[orfs["ribo_evidence"] == e, "priority_score"], bins=30, histtype="step", lw=2, label=e)
axes[1].set(xlabel="priority score", ylabel="ORFs")
axes[1].legend(title="Ribo-seq evidence")
fig.tight_layout()
fig.savefig(FIG / "score_components.pdf")

### 5.6 Positive controls: ZFAS1 and FRG1BP

Both were selected manually for knock-down / CRISPR excision experiments. ZFAS1 ORF3 (13 codons incl. stop)
has peptide support in public HCC cell line proteomics.

In [ ]:
for name in ["ZFAS1", "FRG1BP"]:
    hit = orfs.loc[orfs["locus_name"].astype(str).str.contains(name)
                   | orfs["lncRNA_gene_names"].astype(str).str.contains(name)
                   | orfs["pseudogene_names"].astype(str).str.contains(name)]
    print(f"{name}: {len(hit)} ORFs, best rank {hit['rank'].min() if len(hit) else 'NA'} of {len(orfs)}")
    display(hit[cols].head(10))

### 5.7 Comparison with the manually selected regions

Regions of interest stored in the IGV session used for the manual screen
(`IGV/Asiers_PolySomeSeq.Riboseq.xml`) are compared with the ORFs called here.

In [ ]:
import xml.etree.ElementTree as ET
session = Path(subdirs["main_project_dir"]) / "IGV" / "Asiers_PolySomeSeq.Riboseq.xml"
roi = pd.DataFrame([dict(chrom=r.get("chromosome"), start0=int(r.get("start")), end1=int(r.get("end")))
                    for r in ET.parse(session).getroot().iter("Region")])
def best_overlap(r):
    o = orfs.loc[(orfs["chrom"] == r["chrom"]) & (orfs["orf_start0"] < r["end1"]) & (orfs["orf_end1"] > r["start0"])]
    if o.empty:
        return pd.Series({"n_orfs": 0, "best_rank": np.nan, "best_orf": "", "evidence": "", "locus": ""})
    b = o.sort_values("rank").iloc[0]
    return pd.Series({"n_orfs": len(o), "best_rank": b["rank"], "best_orf": b["orf_label"],
                      "evidence": b["ribo_evidence"], "locus": b["locus_name"]})
roi = pd.concat([roi, roi.apply(best_overlap, axis=1)], axis=1)
if RUN_NAME.startswith("test_"):
    roi = roi.loc[roi["chrom"].isin(set(orfs["chrom"]))]
display(roi)

### 5.8 Overlap with lincRNA ORFs from HCC tissue Ribo-seq (paper)

In [ ]:
if "HCC_paper_lincRNA_ORFs" in orfs.columns:
    orfs["in_HCC_paper"] = orfs["HCC_paper_lincRNA_ORFs"].notna()
    display(pd.crosstab(orfs["ribo_evidence"], [orfs["in_HCC_paper"], orfs["HCC_paper_same_stop"]],
                        margins=True))

## 6. Browsing the candidates in IGV

Open the session in IGV desktop (File > Open Session). It loads:
`lincORFs.igv.gtf` (de novo lincRNA isoforms + ORFs, `transcript_id` = `ORF<rank>_<locus>_<length>aa`),
the ORF BED (colour = evidence class), GENCODE v42, the MRL (fixed 1-10 scale) and polysomal coverage tracks,
pooled Ribo-seq P-sites (CPM), HepG2/Huh7 RNA-seq coverage and the PhyloCSF tracks; + strand in blue, - strand in red.
Clicking on an ORF shows all its metrics, including the amino-acid sequence.
To step through the candidates in rank order, load `lincORFs.loci.ranked.tsv` coordinates or use
`lincORFs.orfs.bed` as Region Navigator input (Regions > Import Regions).

In [ ]:
print(RES / "final" / "lincorfz.igv_session.xml")
print(RES / "final" / "lincORFs.igv.gtf")

## 7. Optional: peptide evidence from small-peptidome data

Non-tryptic (e.g. peptidome or HLA-ligandome) mass-spectrometry data can be searched against
`results/final/lincORFs.faa` together with the canonical human proteome. The resulting peptide tables
(any TSV/CSV with a `Peptide`/`Sequence` column, e.g. FragPipe `peptide.tsv`) are passed to the pipeline as
`"peptide_tables": "/path/a.tsv,/path/b.tsv"`; peptides absent from the GENCODE proteome are matched to
the ORFs and add the peptide component to the priority score. Re-submitting the run with `-resume`
recomputes only the affected steps.

Candidate public non-tryptic datasets (to be checked for suitability before use):
* [PXD023143](https://proteomecentral.proteomexchange.org/dataset/PXD023143): HLA immunopeptidome of isolated
  patient-derived hepatocytes (HBV infection, HCC); already part of the PRIDE re-analysis of the HCC paper,
  but there searched with tryptic settings.
* HCC immunopeptidomics used by [Camarena et al., Sci Adv 2024](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11235171/)
  (ref. 31 of the HCC paper), which reported lncRNA-derived HLA ligands in HCC; see their data availability section.

Such data must be searched with unspecific cleavage (e.g. FragPipe "Nonspecific-HLA" workflow) against
`lincORFs.faa` + the human proteome.

In [ ]:
# example: add peptide tables to the v1 parameters and resume
# p = json.load(open(PARAMS_DIR / "v1.json"))
# p["peptide_tables"] = ",".join(["/path/to/peptidome1/peptide.tsv", "/path/to/peptidome2/peptide.tsv"])
# json.dump(p, open(PARAMS_DIR / "v1.json", "w"), indent=4)
# print(launch_command("v1"))

## 8. Execute this notebook as a SLURM job

To (re-)generate all outputs and figures of this notebook without computing on a login/VS Code node.

In [ ]:
nb_path = PROJECT_DIR / "Liver_lincRNA_ORFs.ipynb"
logs = OUT_BASE / "slurm_logs"
logs.mkdir(exist_ok=True)
print(f"sbatch --output={logs}/notebook_%j.out --error={logs}/notebook_%j.err "
      f"{SBATCH_DIR}/execute_notebook.sbatch {nb_path} {CONDA_ENVS_DIR / 'lincorfz_py'}")